# BasaQuest Phase 3 - Fine-tune a Filipino phoneme recognizer

Part of the "Filipino Local Mispronunciation Detector" project - see the root `ROADMAP.md` and `training/README.md` in the BasaQuest Prototype repo for full context.

Fine-tunes `facebook/wav2vec2-xls-r-300m` (a pretrained multilingual acoustic model) as a **phoneme recognizer** - CTC over the Filipino phoneme inventory from Phase 1's G2P - on the filtered, phoneme-labeled corpus from Phase 2. Its output feeds Phase 4 (GOP scoring).

## ✅ STATUS: Phase 3 is COMPLETE (2026-07-22)

**Final model: `My Drive/BasaQuest/final-model-v2` — full-val PER 0.2968** (measured on the complete 8,831-row validation set). That model is what Phase 4 loads. An earlier save at `final-model` scored 0.3015 and is superseded.

**Training has converged at this configuration.** Validation loss bottomed out at 0.2454 and did not improve across 3,000 further steps; PER sat in a 0.2915-0.3017 band with no trend. **Running more epochs on this setup is wasted effort.** The levers that would actually improve accuracy, in order of impact:

1. **Child speech data.** Everything here is adult speech; the real users are Grade 1-6 pupils. Shorter vocal tracts shift formants upward, so vowel recognition degrades — and for GOP scoring the risk is *false positives*, flagging correctly-spoken child audio as mispronounced.
2. **A bigger model** — `facebook/wav2vec2-xls-r-1b`, same data and code, mostly a `MODEL_NAME` change. Viable now that deployment was decided as server-side rather than on-tablet. Needs a smaller batch + gradient checkpointing on a T4, or A100 time.
3. **Fixing the Phase 1 G2P's known label errors** (penultimate-stress default, unrecovered word-medial/final glottal stops). Wrong training targets are a hard ceiling that no model size fixes.

## Before you start

1. In [drive.google.com](https://drive.google.com), in "My Drive", create a folder named exactly `BasaQuest`, open it, and upload into it:
   - **`basaquest-processed.tar`** - a single-file archive of `training/processed/` (train/val/test parquet shards, ~2.8GB). Packed as one file on purpose: 156 individual small parquet files is a common source of stalled/partial uploads through Drive's web UI; one file isn't. A cell below unpacks it automatically once it's in Drive.
   - **`g2p/`** - the whole folder, uploaded as-is (small, ~90KB, no packing needed). This notebook imports `filipino_g2p` directly from it, so the phoneme vocabulary always matches Phase 1/2 exactly - no copy-pasted vocab list to drift out of sync.

   End result in Drive: `My Drive/BasaQuest/basaquest-processed.tar` and `My Drive/BasaQuest/g2p/filipino_g2p/*.py`. The "Unpack the uploaded corpus archive" cell below turns the `.tar` into local `/content/processed/{train,val,test}/*.parquet` - you don't need to unpack it by hand.

2. **Runtime > Change runtime type > GPU** (T4 works; A100 is ~8x faster but burns ~15 of Colab Pro's ~100 monthly compute units per hour vs T4's ~1.76).

3. Run the cells top to bottom.

**To continue training an existing model**, run Config → model-load → Train. The Config cell must run first — it generates a fresh timestamped `SESSION_DIR`, without which the checkpoint-rotation bug returns.

**Colab Pro does NOT have background execution** (that is Pro+ only) — the browser tab must stay connected. But checkpoints save every 500 steps to a collision-proof folder, so a disconnect costs ~7 minutes, not the run.

In [ ]:
!nvidia-smi

## Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Utility: permanently empty Drive trash (run whenever you need space back)

`save_total_limit` rotates old checkpoints out, but "deleted" via the Drive API just means moved to Trash, not actually freed - space stays used until Trash is emptied. This calls the Drive API's `emptyTrash` directly (not the same permission as reading/writing files - authenticates separately via `auth.authenticate_user()`, one extra login prompt the first time you run it). Re-runnable any time, safe - only removes things already deleted, doesn't touch anything currently in checkpoints/final/g2p/etc.

In [ ]:
!pip install -q google-api-python-client

from google.colab import auth
from googleapiclient.discovery import build

auth.authenticate_user()
drive_service = build("drive", "v3")
drive_service.files().emptyTrash().execute()
print("Drive trash permanently emptied.")

## Optional: SSH tunnel for Claude Code (run this now, BEFORE the Train cell below - or skip it)

Sets up an SSH tunnel into this VM so Claude Code can run from your own local terminal. Legitimate on Colab Pro with a positive compute balance (Google's free-tier "no remote shells" restriction specifically lifts under that condition).

**Why timing matters:** the code cell below goes through this notebook's normal execution queue, same as every other cell. If you try to run it *after* the Train cell is already executing, it will sit queued and do nothing until training finishes or is interrupted - it will not corrupt or break training, but it also won't take effect while training is running. Run this cell now, before Train, if you want SSH access during your training run. Once the tunnel is up, it keeps running in the background independently of the kernel queue, so Train can run afterward without this interfering.

If training is already running: skip this and use the **Terminal tab at the bottom of the Colab window** instead - that's a separate shell outside the kernel entirely, so it's always safe regardless of what's currently executing.

In [ ]:
!pip install -q colab-ssh --upgrade
from colab_ssh import launch_ssh_cloudflared

launch_ssh_cloudflared(password="pick-a-real-password-here")

## Install dependencies

Deliberately does **not** install `torch` - Colab's preinstalled torch is already built against the runtime's CUDA version, and pip-installing a fresh one is a common way to break that match.

In [ ]:
!pip install -q transformers accelerate datasets evaluate jiwer soundfile

## Config

`QUICK_TEST=True` trains on a small slice for a few steps, just to prove the whole pipeline (data loading -> tokenizer -> model -> training loop -> checkpoint save) runs without error before committing to a multi-hour real run. Switch to `False` once that's confirmed.

In [ ]:
QUICK_TEST = False   # True only for a fast smoke test on a tiny slice
QUICK_TEST_TRAIN_ROWS = 500
QUICK_TEST_VAL_ROWS = 100

MODEL_NAME = "facebook/wav2vec2-xls-r-300m"

DRIVE_ROOT     = "/content/drive/MyDrive/BasaQuest"
DATA_DIR       = "/content/processed"
G2P_DIR        = f"{DRIVE_ROOT}/g2p"
PROCESSOR_DIR  = f"{DRIVE_ROOT}/processor"

# Root holding ALL checkpoints, old sessions included. Resume scans this whole
# tree by modification time, so your latest real weights are always found no
# matter which session (or reset step number) produced them.
CHECKPOINT_ROOT = f"{DRIVE_ROOT}/checkpoints"

# THE core fix for "restarts eat my progress": every launch writes to its own
# unique, timestamped dir. save_total_limit only rotates WITHIN one dir, so a
# fresh run can never delete an older run's checkpoints, and a reset step-count
# (checkpoint-500, 1000, ...) can never collide with an old run's high numbers.
import time
SESSION_DIR    = f"{CHECKPOINT_ROOT}/session-{time.strftime('%Y%m%d-%H%M%S')}"
CHECKPOINT_DIR = SESSION_DIR   # alias so any later cell referencing it still works

# Stable path Phase 4 can always load from, independent of session dirs.
FINAL_DIR = f"{DRIVE_ROOT}/final-model"

print("QUICK_TEST:", QUICK_TEST, "| session:", SESSION_DIR)

## Unpack the uploaded corpus archive

`training/processed/` was packed into a single `basaquest-processed.tar` before upload — 2.8GB across 156 small parquet files is a common source of stalled/partial uploads through Drive's web UI; one file isn't. This unpacks it into **local disk** (`/content/processed`, not Drive) — extracting many small files directly onto Drive's FUSE mount is slow and risks a partial extraction if interrupted. Two distinct things can interrupt it, both observed in practice: the runtime disconnecting mid-way, or the Drive FUSE mount itself dropping mid-read ("Transport endpoint is not connected" - a known Colab+Drive reliability quirk, more likely on a sustained read of one large file, unrelated to plan/account status). Either way this cell verifies the extracted file counts match what's expected before moving on, so a partial extraction fails loudly here instead of surfacing as a confusing `FileNotFoundError` several cells later.

**If you see "Transport endpoint is not connected"**: the Drive mount needs refreshing, not a runtime restart. Run this in a cell first:
```python
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive')
```
Then re-run this unpack cell.

In [ ]:
import os

archive_path = f"{DRIVE_ROOT}/basaquest-processed.tar"
EXPECTED_SHARD_COUNTS = {"train": 139, "val": 1, "test": 16}


def _shard_counts_ok(data_dir):
    if not os.path.isdir(data_dir):
        return False
    for split, expected in EXPECTED_SHARD_COUNTS.items():
        split_dir = os.path.join(data_dir, split)
        if not os.path.isdir(split_dir):
            return False
        actual = len([f for f in os.listdir(split_dir) if f.endswith(".parquet")])
        if actual != expected:
            return False
    return True


if _shard_counts_ok(DATA_DIR):
    print(f"{DATA_DIR} already has all expected shards, skipping unpack.")
elif os.path.exists(archive_path):
    print(f"Unpacking {archive_path} to local disk...")
    os.makedirs("/content", exist_ok=True)
    !tar -xf "{archive_path}" -C /content
    print("Extraction finished, verifying shard counts...")
    for split, expected in EXPECTED_SHARD_COUNTS.items():
        split_dir = os.path.join(DATA_DIR, split)
        actual = len([f for f in os.listdir(split_dir) if f.endswith(".parquet")]) if os.path.isdir(split_dir) else 0
        status = "OK" if actual == expected else "MISMATCH"
        print(f"  {split}: {actual}/{expected} parquet files [{status}]")
    if not _shard_counts_ok(DATA_DIR):
        raise RuntimeError(
            "Extraction is incomplete - shard counts don't match what Phase 2 produced. "
            "Two known causes: (1) the runtime disconnected mid-extraction - reconnect and "
            "re-run this cell; (2) the Drive FUSE mount dropped mid-read (tar errors above "
            "would say 'Transport endpoint is not connected' if so) - refresh it first with "
            "drive.flush_and_unmount() followed by drive.mount('/content/drive'), THEN "
            "re-run this cell. If it keeps happening after both, re-upload "
            "basaquest-processed.tar to Drive in case the upload itself was truncated."
        )
    print("All shards verified present.")
else:
    print(f"No archive found at {archive_path} and {DATA_DIR} is empty/missing - upload basaquest-processed.tar to Drive first (see training/README.md).")

## Build the phoneme tokenizer

Imports `filipino_g2p` directly from the `g2p/` folder you uploaded, so the CTC vocabulary is generated from Phase 1's actual code, not a hand-copied list that could drift out of sync with it.

`Wav2Vec2PhonemeCTCTokenizer` (not the plain `Wav2Vec2CTCTokenizer`) is the right class here: it treats **space-separated tokens as whole units** rather than splitting text character-by-character, which is exactly what a multi-character phoneme like `"ng"` or `"a1"` needs. `do_phonemize=False` is important - without it, the tokenizer tries to run its own (English-oriented, espeak-based) phonemizer on the input, which would double-process text we've already turned into phonemes in Phase 2.

In [ ]:
import sys, json, os

sys.path.insert(0, G2P_DIR)
from filipino_g2p import full_vocabulary

phoneme_vocab = full_vocabulary()
vocab_dict = {p: i for i, p in enumerate(phoneme_vocab)}
vocab_dict["|"] = len(vocab_dict)      # word delimiter - required by the tokenizer class, unused by our flat phoneme_sequence targets
vocab_dict["<unk>"] = len(vocab_dict)
vocab_dict["<pad>"] = len(vocab_dict)

print(f"{len(vocab_dict)} tokens ({len(phoneme_vocab)} phonemes + 3 special)")

os.makedirs("/content/vocab", exist_ok=True)
with open("/content/vocab/vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab_dict, f, ensure_ascii=False)

from transformers import Wav2Vec2PhonemeCTCTokenizer

tokenizer = Wav2Vec2PhonemeCTCTokenizer(
    "/content/vocab/vocab.json",
    unk_token="<unk>",
    pad_token="<pad>",
    word_delimiter_token="|",
    do_phonemize=False,
)

In [ ]:
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Processor

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1, sampling_rate=16000, padding_value=0.0, do_normalize=True, return_attention_mask=True
)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
processor.save_pretrained(PROCESSOR_DIR)
print("saved processor to", PROCESSOR_DIR)

## Load the Phase 2 output

`datasets.load_dataset('parquet', ...)` reads the shards directly; casting the `audio` column to an `Audio(16000)` feature makes each row's audio bytes decode to a waveform array lazily, only when actually accessed (not all up front).

In [ ]:
from datasets import load_dataset, Audio

raw = load_dataset(
    "parquet",
    data_files={
        "train": f"{DATA_DIR}/train/*.parquet",
        "val": f"{DATA_DIR}/val/*.parquet",
    },
)
raw = raw.cast_column("audio", Audio(sampling_rate=16000))

# Cap clip duration: a small tail of clips runs up to 8+ seconds while most
# are under 2s (see Phase 2's corpus sampling in training/README.md).
# Transformer self-attention memory cost scales up sharply, not linearly,
# with sequence length - so a batch that happens to contain several of the
# longest clips together can spike memory far past what a typical batch
# needs. That's what caused a CUDA OOM crash partway through an earlier run.
MAX_DURATION_SECONDS = 6.0

# Drop clips too SHORT for their own phoneme target. Wav2Vec2's CNN feature
# encoder downsamples audio by a fixed ~320x (16kHz -> ~50 output frames per
# second - the standard, documented rate for this architecture family). CTC
# loss needs at least one output frame per target phoneme, realistically
# more. A clip with too few frames for its phoneme count produces an
# invalid (sometimes literally negative) length and crashes ctc_loss - this
# is what actually caused "Validation Loss: inf" from step 200 onward in
# every run so far, silently, until it finally crashed training outright
# instead of just corrupting one eval batch's number. The 2x margin below
# accounts for this being an approximation (not the exact conv formula) and
# for CTC needing extra frames around repeated adjacent phonemes.
MIN_FRAMES_PER_PHONEME = 2.0
APPROX_FRAME_RATE = 50.0

before = {split: len(raw[split]) for split in raw}
raw = raw.filter(
    lambda x: x["duration"] <= MAX_DURATION_SECONDS
    and (x["duration"] * APPROX_FRAME_RATE) >= (x["phoneme_count"] * MIN_FRAMES_PER_PHONEME)
)
for split in raw:
    dropped = before[split] - len(raw[split])
    print(f"{split}: dropped {dropped} clip(s) (too long, or too short for their phoneme count) - {len(raw[split])} remain")

if QUICK_TEST:
    raw["train"] = raw["train"].select(range(min(QUICK_TEST_TRAIN_ROWS, len(raw["train"]))))
    raw["val"] = raw["val"].select(range(min(QUICK_TEST_VAL_ROWS, len(raw["val"]))))

print(raw)

## Prepare examples: audio -> input_values, phoneme_sequence -> labels

In [ ]:
def prepare_batch(batch):
    audio = batch["audio"]
    batch["input_values"] = processor.feature_extractor(audio["array"], sampling_rate=16000).input_values[0]
    batch["input_length"] = len(batch["input_values"])
    # phoneme_sequence is already a list of phoneme tokens (Phase 2 output) - join with the
    # tokenizer's expected phone delimiter (a plain space) rather than re-deriving phonemes here.
    batch["labels"] = processor.tokenizer(" ".join(batch["phoneme_sequence"])).input_ids
    return batch


encoded = raw.map(prepare_batch, remove_columns=raw["train"].column_names, num_proc=1)

## Data collator (pads a batch, masks label padding for CTC loss)

In [ ]:
import torch
from dataclasses import dataclass
from typing import Dict, List, Union


@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.feature_extractor.pad(input_features, padding=self.padding, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, padding=self.padding, return_tensors="pt")

        # CTC loss should ignore padding positions in the labels - -100 is the
        # standard "ignore this position" sentinel for HF loss functions.
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch


data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

## Load the latest checkpoint (robust resume) + add the CTC head

This is the resume logic, rewritten to fix the bug where restarts silently lost progress. Instead of `get_last_checkpoint` (which picks by the number in the folder name — unreliable once step counters reset per session), it scans the **whole** `CHECKPOINT_ROOT` and loads the newest checkpoint by **modification time**, walking newest-first and skipping any half-written/corrupt one from an interrupted save. That always lands on your most-trained weights, whatever they're numbered. If nothing loadable is found, it falls back to the base `MODEL_NAME`.

`freeze_feature_encoder()` keeps the pretrained CNN feature extractor fixed and only fine-tunes the transformer + CTC head — the standard, less overfitting-prone default.

In [ ]:
import os, glob, time, torch
from transformers import Wav2Vec2ForCTC


def list_checkpoints_newest_first(root):
    cks = [c for c in glob.glob(os.path.join(root, "**", "checkpoint-*"), recursive=True)
           if os.path.isdir(c)]
    cks.sort(key=lambda c: os.path.getmtime(c), reverse=True)
    return cks


model = None
for ckpt in list_checkpoints_newest_first(CHECKPOINT_ROOT):
    try:
        print("Loading latest checkpoint:", ckpt,
              "(saved", time.ctime(os.path.getmtime(ckpt)) + ")")
        model = Wav2Vec2ForCTC.from_pretrained(
            ckpt,
            ctc_loss_reduction="mean",
            pad_token_id=processor.tokenizer.pad_token_id,
            vocab_size=len(processor.tokenizer),
        )
        print("  loaded OK -> continuing training from here")
        break
    except Exception as e:
        print("  skipped (incomplete/corrupt), trying an older one:", e)

if model is None:
    print("No usable checkpoint found -> starting fresh from", MODEL_NAME)
    model = Wav2Vec2ForCTC.from_pretrained(
        MODEL_NAME,
        ctc_loss_reduction="mean",
        pad_token_id=processor.tokenizer.pad_token_id,
        vocab_size=len(processor.tokenizer),
    )

model.freeze_feature_encoder()

## Metric: Phoneme Error Rate

The generic `wer` metric is just token-edit-distance over whitespace-split tokens - applied to our space-separated phoneme strings, it *is* phoneme error rate (PER). Same trick used by HF's own phoneme-recognition model cards (e.g. `wav2vec2-lv-60-espeak-cv-ft`).

In [ ]:
import numpy as np
import evaluate

wer_metric = evaluate.load("wer")


def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    per = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"per": per}

## Train (robust + fast)

Rewritten after a full day of real runs on 2026-07-22. Every choice here exists because something broke:

- **Writes to a unique per-session `SESSION_DIR`** (set in Config). This is the fix for the checkpoint-rotation bug: `save_total_limit` only ever deletes within the current session's own dir, so a fresh run can never delete an older run's checkpoints, and a reset step number can't collide with an old high number. Before this fix, every restart's new checkpoints were silently deleted the instant they were written, because they were numerically smaller than a stale leftover.
- **No `resume_from_checkpoint`** — the latest weights were already loaded in the model cell above. Fresh `Trainer` + new dir sidesteps both the old eval-schedule bug and the rotation bug.
- **GPU-aware batch sizing**, auto-detected: A100 gets a big batch; T4 falls back to the OOM-safe `4 × grad_accum 4`.
- **Early stopping on `eval_loss`, patience 6.** It was originally `per`/patience 4 — that killed a run at step 5500 while validation loss was still hitting new lows at every eval. PER is computed on a 2k-row subset and is too noisy to steer a stopping rule.
- **2k-row subset eval** in-loop for speed (the full 8,831-row eval took ~157s *each*); the honest full-val number is printed by the save cell.
- Lower `learning_rate=1e-4` and `warmup_steps=0` because this continues an already-converging model rather than starting cold.

**Re-running to continue training?** Re-run the **Config cell first** so `SESSION_DIR` gets a fresh timestamp, then the model-load cell, then this one. Skipping Config reuses the old session folder and — because the step counter restarts at 0 — resurrects the rotation bug.

In [ ]:
import torch
from transformers import (TrainingArguments, Trainer, TrainerCallback,
                          EarlyStoppingCallback)
from google.colab import auth
from googleapiclient.discovery import build

# --- GPU-aware batch sizing (auto-detected; falls back safely on T4) --------
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
if "A100" in gpu:
    bs, accum = 24, 1          # 40GB: big batch, no accumulation needed
elif "L4" in gpu or "V100" in gpu:
    bs, accum = 12, 1
else:
    bs, accum = 4, 4           # T4 / unknown: the config that survived the OOM
print(f"GPU={gpu}  ->  batch={bs}  grad_accum={accum}  (effective {bs * accum})")

# Fast in-loop signal on a 2k-row subset (the full 8,831-row eval took ~157s
# each and dominated wall-clock). Real full-val number is printed at the end
# by the save cell.
eval_subset = encoded["val"].select(range(min(2000, len(encoded["val"]))))
STEPS = 200 if QUICK_TEST else 500


class EmptyTrashCallback(TrainerCallback):
    """Empties Drive trash after each save so 'deleted' rotated checkpoints
    actually free space instead of piling up in Trash. Run the 'empty Drive
    trash' utility cell once up front so the login is cached and this won't
    prompt mid-training."""

    def __init__(self):
        self._svc = None

    def _service(self):
        if self._svc is None:
            auth.authenticate_user()
            self._svc = build("drive", "v3")
        return self._svc

    def on_save(self, args, state, control, **kwargs):
        try:
            self._service().files().emptyTrash().execute()
            print(f"[EmptyTrash] step {state.global_step}")
        except Exception as e:
            print(f"[EmptyTrash] non-fatal: {e}")


training_args = TrainingArguments(
    output_dir=SESSION_DIR,                 # unique per launch -> rotation is safe
    per_device_train_batch_size=bs,
    per_device_eval_batch_size=bs,
    gradient_accumulation_steps=accum,
    eval_strategy="steps",
    eval_steps=STEPS,
    save_steps=STEPS,
    save_total_limit=2,
    num_train_epochs=(1 if QUICK_TEST else 3),   # continuing an already-trained model
    learning_rate=1e-4,                     # lower LR: continuing a converging model
    warmup_steps=0,                         # already past warmup; don't re-ramp from 0
    fp16=True,
    logging_steps=50,
    load_best_model_at_end=True,
    # Stop on validation loss, NOT PER. Learned the hard way 2026-07-22: PER is
    # computed on the 2k-row subset and is far too noisy to steer early
    # stopping — it flatlined by chance across steps 3500-5500 and killed a run
    # while eval_loss was still setting new lows at every single eval.
    metric_for_best_model="eval_loss",
    greater_is_better=False,                # lower loss = better
    # DO NOT add save_safetensors here — it is not a valid kwarg in the
    # installed transformers version and raises TypeError. Safetensors is the
    # default there anyway. Same trap as the removed group_by_length.
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    train_dataset=encoded["train"],
    eval_dataset=eval_subset,
    processing_class=processor.feature_extractor,
    callbacks=[EmptyTrashCallback(),
               EarlyStoppingCallback(early_stopping_patience=6)],
)

# No resume_from_checkpoint on purpose: weights were already loaded in the
# model cell above. Fresh Trainer + brand-new SESSION_DIR is what avoids BOTH
# the old eval-schedule bug and the checkpoint-rotation bug.
trainer.train()

## Save the final model + processor

In [ ]:
# load_best_model_at_end already restored the best-PER checkpoint into `trainer`.
trainer.save_model(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)
print("Saved final model + processor to", FINAL_DIR)

# The real number: full validation set (not the 2k subset used during training).
full_val_per = trainer.evaluate(encoded["val"])["eval_per"]
print("Full-val PER:", full_val_per)

## Sanity check: listen to what it learned

Runs the model on one validation example and prints predicted vs. reference phonemes, so you can eyeball whether it's learning something sensible before moving to Phase 4 (GOP scoring). This is a smell test, not a substitute for the real Phase 5 evaluation.

In [ ]:
model.eval()
sample = encoded["val"][0]

with torch.no_grad():
    input_values = torch.tensor(sample["input_values"]).unsqueeze(0).to(model.device)
    logits = model(input_values).logits

pred_ids = torch.argmax(logits, dim=-1)
pred_str = processor.batch_decode(pred_ids)[0]

label_ids = [i if i != -100 else processor.tokenizer.pad_token_id for i in sample["labels"]]
label_str = processor.batch_decode([label_ids], group_tokens=False)[0]

print("reference: ", label_str)
print("predicted: ", pred_str)

## Next steps

Phase 3 is done — see the status block at the top. `My Drive/BasaQuest/final-model-v2` (full-val PER 0.2968) is what Phase 4 loads.

1. **Phase 4 — GOP scoring.** Download `final-model-v2` from Drive (~1.2GB) and build it locally: GOP inference is a single forward pass over one short clip, so it runs fine on CPU and doesn't need Colab. Force-align the expected phoneme sequence (from Phase 1's G2P) to the audio with `torchaudio.functional.forced_align`, score each expected phoneme against its best competitor, aggregate to word level, threshold, and emit the same shape `scorePassage()` already returns.
2. **Phase 5 — the real evaluation.** Run the same real recordings through both this and the existing `whisperScorer.js`, then judge by ear which flags are more correct. This, not PER, decides whether the project worked.
3. **Only then** consider more training. Get real child recordings first and measure PER on them — the adult/child acoustic gap is the largest unknown, and it changes what's worth optimizing.

**If you do train more**, run Config → model-load → Train (Config first, always), and set `FINAL_DIR` to a new path in a cell above the save cell so you don't overwrite a known-good model with a worse one.